# Lecture 05 — Files

*The everyday chemist's task: open a data file, process it, and save the result.*

## Learning objectives

By the end of this lecture you will be able to:

- explain what a **file path** is, including what `..` means;
- open a file safely with a **`with`** block, and read it (`.read()`, `.readlines()`, line by line);
- **write** text to a new file;
- understand **CSV** as comma-separated text, and read it with both the `csv` module and `pandas`;
- handle a missing file gracefully with **`try` / `except`**.

## Recap of Lecture 04

- **`if` / `elif` / `else`** make decisions; **`for`** and **`while`** loops repeat work.
- We built a `rule_of_five()` screen and looped it over several molecules.
- A **list comprehension** is shorthand for a list-building loop.

## What is a file path?

A **file path** is the address of a file on your computer. Our data lives in the project's `data/` folder, while this notebook lives in `notebooks/`. To reach the data from here we go *up one level* and then into `data/`:

```
../data/molecules.csv
```

The `..` means "the folder above this one" (here, the project root). The `/` separates folder names. We will use this path throughout the lecture.

## Reading a text file with `with`

The safe way to open a file is a **`with`** block:

```python
with open(path) as f:
    ... use f ...
```

The reason to use `with`: it **automatically closes** the file for you when the block ends, even if something goes wrong. Leaving files open is a classic source of subtle bugs, so always use `with`.

Let us read the molecules file as raw text first, just to see what is in it. `.read()` slurps the whole file into one string; here we print only the first 200 characters.

In [ ]:
path = "../data/molecules.csv"

with open(path) as f:
    contents = f.read()

print(contents[:200])      # the first 200 characters

That is a **CSV** file — *comma-separated values*. The first line is a **header** naming the columns; each line after is one molecule, with fields separated by commas.

Often you want the file **line by line** instead of as one blob. `.readlines()` gives you a list of lines; or you can loop straight over the file object.

In [ ]:
with open(path) as f:
    lines = f.readlines()

print("Number of lines (including header):", len(lines))
print("Header:", lines[0].strip())          # .strip() removes the trailing newline
print("First molecule row:", lines[1].strip())

### 🔬 Try it yourself

Open `../data/molecules.csv`, loop over the lines **after** the header (remember slicing — `lines[1:]`), and print just the **first field** (the name) of each row. Hint: `line.split(",")` splits a line into a list of fields.

In [ ]:
# Your code here.

**Solution**

In [ ]:
with open(path) as f:
    lines = f.readlines()

for line in lines[1:]:            # skip the header
    fields = line.strip().split(",")
    print(fields[0])              # the name is the first field

## Reading a CSV properly: the `csv` module

Splitting on commas by hand is fragile. Python's built-in **`csv`** module does it correctly. `csv.DictReader` is especially friendly: it reads each row into a **dictionary** keyed by the column names — exactly the data structure from Lecture 02.

In [ ]:
import csv

with open(path, newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)          # turn it into a list of dictionaries

print("Read", len(rows), "molecules")
print(rows[0])                   # the first row, as a dictionary
print("First molecule's name:", rows[0]["name"])

Note that everything read from a text file arrives as a **string** — even the numbers. To do arithmetic we must convert with `float(...)`. Let us report each molecule's molar mass:

In [ ]:
for row in rows:
    mass = float(row["molar_mass"])      # convert the text "18.02" to the number 18.02
    print(f"{row['name']:<12} {mass:7.2f} g/mol")

## Applying the rule-of-five filter and writing a new file

Now the real task. We will reuse the rule-of-five idea from Lecture 04 — but this time the descriptors are already in the file, so we just read them. We collect the drug-like molecules and **write them to a new CSV**.

In [ ]:
def passes_rule_of_five(row):
    "Decide drug-likeness from a CSV row (values are strings, so convert them)."
    violations = 0
    if float(row["molar_mass"]) > 500:
        violations += 1
    if float(row["logp"]) > 5:
        violations += 1
    if int(row["h_bond_donors"]) > 5:
        violations += 1
    if int(row["h_bond_acceptors"]) > 10:
        violations += 1
    return violations <= 1

drug_like = [row for row in rows if passes_rule_of_five(row)]
print(f"{len(drug_like)} of {len(rows)} molecules pass the rule of five")

In [ ]:
out_path = "../data/drug_like_subset.csv"

with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(drug_like)

print("Wrote", out_path)

Open the `"w"` argument up for a moment: it means **write** mode (it creates the file, or overwrites it if it exists). Reading mode is the default. Be careful — `"w"` will happily wipe an existing file.

## The easier way: pandas

Reading CSVs is so common that the **pandas** library makes it a one-liner. Think of a pandas **DataFrame** as *a smart table* — like a spreadsheet you can compute with. We will not turn this into a pandas course; just meet it as a convenient tool for tabular data.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/molecules.csv")
df          # a DataFrame displays as a tidy table

pandas already knows the numeric columns are numbers, so you can compute directly — no `float(...)` needed:

In [ ]:
print("Mean molar mass:", round(df["molar_mass"].mean(), 2), "g/mol")
print("Heaviest molecule:", df.loc[df["molar_mass"].idxmax(), "name"])

# The same drug-like filter, the pandas way:
mask = (df["molar_mass"] <= 500) & (df["logp"] <= 5) & \
       (df["h_bond_donors"] <= 5) & (df["h_bond_acceptors"] <= 10)
print(df[mask]["name"].tolist())

## Handling a missing file: `try` / `except`

If you ask to open a file that is not there, Python raises an error and stops. Often you would rather **catch** that and carry on. A **`try` / `except`** block does this: Python *tries* the risky code, and if the named error happens, it runs the *except* block instead of crashing.

In [ ]:
try:
    with open("../data/does_not_exist.csv") as f:
        f.read()
except FileNotFoundError:
    print("That file is not there — check the path and try again.")

## ⚗️ With RDKit — confirm the file's numbers from the SMILES

The file *claims* certain molar masses. We can verify them: take the `smiles` column, rebuild each molecule with RDKit, recompute the molar mass, and compare. This is exactly how `data/molecules.csv` was generated in the first place.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

for row in rows[:5]:                      # just the first five, to keep it short
    mol = Chem.MolFromSmiles(row["smiles"])
    from_rdkit = Descriptors.MolWt(mol)
    from_file = float(row["molar_mass"])
    agree = abs(from_rdkit - from_file) < 0.1
    print(f"{row['name']:<12} file={from_file:7.2f}  rdkit={from_rdkit:7.2f}  match={agree}")

> **🧪 Chemistry aside — other file formats.** CSV is fine for tables of properties, but chemists also use dedicated structure formats such as **SDF** and **MOL**, which store full 2D/3D atom-and-bond information. RDKit can read these too (e.g. `Chem.SDMolSupplier`) — worth knowing they exist; no need to dive in now.

### 🔬 Try it yourself

1. Count how many molecules in `rows` pass the rule of five (you have `passes_rule_of_five`).
2. Write a short **summary text file** to `../data/summary.txt` containing one line: how many of how many molecules are drug-like. Read it back and print it to check.

In [ ]:
# Your code here.

**Solution**

In [ ]:
n_pass = sum(1 for row in rows if passes_rule_of_five(row))

with open("../data/summary.txt", "w") as f:
    f.write(f"{n_pass} of {len(rows)} molecules are drug-like by Lipinski's rule of five.\n")

with open("../data/summary.txt") as f:
    print(f.read())

## Key takeaways

- A **file path** locates a file; `..` means the folder above.
- Always open files in a **`with`** block — it closes them for you automatically.
- Read text with `.read()` / `.readlines()`; write with mode `"w"` (which overwrites!).
- **CSV** is comma-separated text; read it cleanly with the `csv` module or, even more easily, `pandas.read_csv` (a DataFrame is "a smart table").
- Catch errors like a missing file with **`try` / `except`**.

## Looking ahead

Next lecture — **NumPy** — we move from rows-and-loops to fast **arrays**, doing maths on whole columns of numbers at once (and meeting the Beer–Lambert law).